# LangChain Expression Language (LCEL) with Groq

**LCEL** is LangChain's declarative way to connect components such as prompts, models, parsers, retrievers, and ordinary Python functions.

## What you will learn

- Build a modern `prompt | model | parser` chain.
- Understand `Runnable` and the pipe operator `|`.
- Use `invoke()`, `stream()`, and `batch()`.
- Add input validation and normalization.
- Add retries and runtime metadata.
- Run independent chains with `RunnableParallel`.
- Understand why legacy `LLMChain` is no longer needed.

> Select the **Python 3.13.7 (`.venv`)** kernel and run the cells from top to bottom.

## 1. Production-style configuration

### What?

Configuration controls the provider model, response length, timeout, and retry count.

### Why?

Enterprise applications keep settings outside chain logic. Typed validation catches configuration errors at startup instead of during a user request.

Your `.env` can contain:

```text
GROQ_API_KEY=your_groq_api_key
GROQ_MODEL=qwen/qwen3.8-27b
GROQ_MAX_TOKENS=256
GROQ_TIMEOUT_SECONDS=30
GROQ_MAX_RETRIES=2
```

Never commit a real API key.

In [1]:
import logging
import os
from dataclasses import dataclass

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True,
)
logger = logging.getLogger("lcel_demo")


def positive_int_from_env(name: str, default: int) -> int:
    raw_value = os.getenv(name, str(default))
    try:
        value = int(raw_value)
    except ValueError as error:
        raise ValueError(f"{name} must be an integer, received {raw_value!r}.") from error
    if value <= 0:
        raise ValueError(f"{name} must be greater than zero.")
    return value


@dataclass(frozen=True, slots=True)
class Settings:
    model: str
    max_output_tokens: int
    timeout_seconds: int
    max_retries: int

    @classmethod
    def from_environment(cls) -> "Settings":
        if not os.getenv("GROQ_API_KEY"):
            raise RuntimeError("GROQ_API_KEY is missing from the .env file.")
        return cls(
            model=os.getenv("GROQ_MODEL", "qwen/qwen3.8-27b"),
            max_output_tokens=positive_int_from_env("GROQ_MAX_TOKENS", 256),
            timeout_seconds=positive_int_from_env("GROQ_TIMEOUT_SECONDS", 30),
            max_retries=positive_int_from_env("GROQ_MAX_RETRIES", 2),
        )


settings = Settings.from_environment()
logger.info("LCEL configuration loaded for model %s", settings.model)

2026-10-01 13:10:56,637 | INFO | lcel_demo | LCEL configuration loaded for model qwen/qwen3.8-27b


## 2. What is a Runnable?

A **Runnable** is a LangChain component with a standard execution interface. Prompts, chat models, parsers, retrievers, and LCEL chains are all runnables.

Common methods:

| Method | Use it when | Result |
| --- | --- | --- |
| `invoke(input)` | Processing one input | One output |
| `stream(input)` | Displaying output incrementally | Iterator of chunks |
| `batch(inputs)` | Processing several independent inputs | List of outputs |
| `ainvoke(input)` | Working inside asynchronous code | Awaitable output |

Because components share this interface, they can be composed without writing orchestration boilerplate.

In [2]:
from langchain_groq import ChatGroq

chat_model = ChatGroq(
    model=settings.model,
    temperature=0.2,
    reasoning_effort="none",
    max_tokens=settings.max_output_tokens,
    timeout=settings.timeout_seconds,
    max_retries=settings.max_retries,
)

## 3. Build a basic LCEL chain

```text
input dictionary → ChatPromptTemplate → ChatGroq → StrOutputParser → string
```

### Components

| Component | What it does | Why use it? |
| --- | --- | --- |
| `ChatPromptTemplate` | Converts variables into structured chat messages | Keeps prompts reusable and avoids manual string construction |
| `ChatGroq` | Sends messages to the selected Groq model | Generates the response |
| `StrOutputParser` | Extracts text from the model message | Returns a normal Python string |
| `|` | Passes one component's output to the next | Makes the data flow visible |

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

fact_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a concise educational assistant. "
            "Provide one well-known, verifiable fact. Avoid speculation.",
        ),
        ("human", "Tell me one interesting fact about {person}."),
    ]
)

fact_chain = fact_prompt | chat_model | StrOutputParser()

## 4. `invoke()` — process one input

Use `invoke()` when you have one request and need one completed result.

The dictionary key must match the prompt variable `{person}`. The result is a string because the chain ends with `StrOutputParser`.

In [4]:
fact = fact_chain.invoke({"person": "John F. Kennedy"})

print(fact)
print("\nOutput type:", type(fact).__name__)

2026-10-01 13:10:58,104 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


John F. Kennedy was the first U.S. president to be born in the 20th century, having been born on May 29, 1917.

Output type: TextAccessor


## 5. Add input validation with `RunnableLambda`

### What?

`RunnableLambda` turns a normal Python function into an LCEL component.

### When and why?

Use it for deterministic work such as validation, normalization, formatting, routing keys, or adapting data shapes. Validate before an API call so invalid requests fail quickly and do not consume provider quota.

In [5]:
from typing import TypedDict

from langchain_core.runnables import RunnableLambda


class PersonInput(TypedDict):
    person: str


def validate_person_input(data: PersonInput) -> PersonInput:
    person = data.get("person", "").strip()
    if not person:
        raise ValueError("The 'person' value cannot be empty.")
    if len(person) > 100:
        raise ValueError("The 'person' value must not exceed 100 characters.")
    return {"person": person}


validated_fact_chain = (
    RunnableLambda(validate_person_input)
    | fact_prompt
    | chat_model
    | StrOutputParser()
)

In [6]:
print(validated_fact_chain.invoke({"person": "  Nelson Mandela  "}))

2026-10-01 13:10:58,300 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Nelson Mandela was the first person to be elected as the head of state of a country by a universal adult franchise (a vote open to all adults regardless of race) in a fair election.


## 6. Runtime configuration, tracing metadata, and retries

### `with_config()`

Adds an operation name, tags, and metadata. Observability systems such as LangSmith can use these values to filter and diagnose runs.

### `with_retry()`

Repeats a chain after temporary failures. Retries help with transient network or service errors, but they do not fix invalid inputs, exhausted credit, or unsupported model parameters.

Keep retries bounded to avoid duplicate cost and long delays.

In [7]:
reliable_fact_chain = validated_fact_chain.with_retry(
    stop_after_attempt=3,
    wait_exponential_jitter=True,
).with_config(
    {
        "run_name": "generate_person_fact",
        "tags": ["education", "groq", "lcel"],
        "metadata": {"notebook": "06_LCEL"},
    }
)

print(reliable_fact_chain.invoke({"person": "Marie Curie"}))

2026-10-01 13:10:58,481 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Marie Curie is the only person in history to have won Nobel Prizes in two different scientific fields: Physics (1903) and Chemistry (1911).


## 7. `stream()` — display output incrementally

Streaming yields pieces as the model produces them. Use it for chat interfaces or long responses where users should see progress immediately.

- `end=""` prevents `print()` from adding a newline after every chunk.
- `flush=True` immediately sends buffered text to the notebook display.
- The final `print()` moves the cursor to a new line after streaming ends.

In [8]:
for chunk in reliable_fact_chain.stream({"person": "Cristiano Ronaldo"}):
    print(chunk, end="", flush=True)

print()

2026-10-01 13:10:58,567 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Cristiano Ronaldo is the first player in history to score 100 goals in the UEFA Champions League.


## 8. `batch()` — process several independent inputs

`batch()` runs the same chain for multiple inputs and preserves input order in the returned list.

Use it for independent workloads such as classifying records or generating summaries. `max_concurrency` limits simultaneous work so you do not overload the provider or exceed rate limits.

In [9]:
people = [
    {"person": "Lionel Messi"},
    {"person": "Marta Vieira da Silva"},
]

batch_results = reliable_fact_chain.batch(
    people,
    config={"max_concurrency": 2},
)

for input_data, result in zip(people, batch_results, strict=True):
    print(f"\n{input_data['person']}:\n{result}")

2026-10-01 13:10:58,845 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-01 13:10:59,216 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



Lionel Messi:
Lionel Messi is the only player in history to win the FIFA World Player of the Year (and its predecessor, the FIFA World Player of the Year) award a record eight times, achieving this feat between 2009 and 2022.

Marta Vieira da Silva:
Marta Vieira da Silva is the only player in history to win the FIFA Women's World Player of the Year award six times (2006, 2007, 2008, 2009, 2010, and 2018).


## 9. `RunnableParallel` — run independent branches

### What?

`RunnableParallel` sends the same input to multiple runnable branches and returns a dictionary of their results.

### When and why?

Use it when outputs do not depend on each other—for example, generating a fact and a short headline from the same input. Parallel branches can reduce total latency compared with sequential calls.

Be mindful that each model branch is a separate API request and consumes its own quota.

In [10]:
from langchain_core.runnables import RunnableParallel

headline_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Write a neutral headline of at most eight words."),
        ("human", "Create a headline about {person}."),
    ]
)
headline_chain = headline_prompt | chat_model | StrOutputParser()

profile_chain = RunnableLambda(validate_person_input) | RunnableParallel(
    fact=fact_prompt | chat_model | StrOutputParser(),
    headline=headline_chain,
)

profile = profile_chain.invoke({"person": "Ada Lovelace"})
print("Headline:", profile["headline"])
print("Fact:", profile["fact"])

2026-10-01 13:10:59,632 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-01 13:10:59,661 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Headline: Ada Lovelace Pioneers Early Computer Programming
Fact: Ada Lovelace is widely recognized as the first computer programmer because she wrote an algorithm intended to be executed by Charles Babbage's Analytical Engine in 1843.


## 10. LCEL versus legacy `LLMChain`

Older tutorials may use:

```python
LLMChain(llm=model, prompt=prompt)
```

Modern LangChain code uses:

```python
chain = prompt | model | StrOutputParser()
```

| LCEL advantage | Practical value |
| --- | --- |
| One standard `Runnable` interface | The same chain supports invoke, batch, stream, and async execution |
| Visible data flow | Easier to read, test, and debug |
| Easy composition | Add validation, retrieval, parallel branches, parsers, and retries |
| Runtime configuration | Attach tags, metadata, callbacks, and concurrency limits |

## Enterprise checklist

- Store API keys in a managed secret service rather than source code.
- Add tracing, latency, token usage, error-rate metrics, and alerts.
- Bound retries, timeouts, batch concurrency, input size, and output tokens.
- Validate and sanitize external input before prompts or tools.
- Use structured output for machine-to-machine workflows.
- Evaluate prompts and models against a versioned test dataset before deployment.
- Confirm that data sent to an external model provider is approved for that provider.